# NexusAI — Production Fine-Tuning (Step 1 → Step 12)

Llama-3 8B ko **Unsloth + LoRA (r=32) + SFTTrainer** se fine-tune karta hai tumhare `nexusai_train.jsonl` (5000 messages) par.

**Settings:** Stage 2 Production — 2 epochs, cosine LR, packing ON, checkpoints every 250 steps.

**Rules:**
1. `import unsloth` SABSE PEHLE — `trl`/`transformers`/`peft` se before.
2. `processing_class=tokenizer` (`tokenizer=` deprecated).
3. `SFTConfig` use karo (`TrainingArguments` nahi).
4. Step 1 ke baad **Runtime → Restart session** ZAROORI.
5. Step 4 me **Helper cell pehle** → fir **4A** (upload).

## Step 1 — Install dependencies

Iske baad **Runtime → Restart session** ZAROORI hai.

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install --upgrade --no-cache-dir "transformers>=4.46.0" "trl>=0.12.0" "peft>=0.13.0" "accelerate>=1.0.0" "bitsandbytes>=0.44.0" "datasets>=2.20.0"
print("Done. Ab RESTART karo: Runtime -> Restart session")

## Step 2 — Load model + tokenizer (4-bit quantized)

In [ ]:
import unsloth  # <-- MUST be first import
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

max_seq_length = 2048
dtype = None            # auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)
print("Model loaded:", model.config._name_or_path)

## Step 3 — Attach LoRA adapters (r=32, production capacity)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,                                # ⬆ upgraded: 16→32 for better learning
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 32,                               # ⬆ match rank
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    use_rslora     = False,
    loftq_config   = None,
)
model.print_trainable_parameters()

## Step 4 — Load dataset

**Pehle Helper cell chalao, fir Step 4A (upload).**

Supported formats: `messages`, `instruction/input/output`, `prompt/completion`, `text`, `question/answer`.

### Helper cell — PEHLE chalao (auto-formatter + chat template fix)

In [ ]:
from datasets import load_dataset, Dataset
from unsloth.chat_templates import get_chat_template

# Llama-3 base model me chat_template missing hota hai — attach karo
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
print("Chat template set:", tokenizer.chat_template is not None)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def auto_format_dataset(ds):
    """Auto-detect format and create 'text' column."""
    cols = set(ds.column_names)

    if "text" in cols:
        print("[auto-format] 'text' column found, using as-is.")
        return ds

    if {"instruction", "output"}.issubset(cols):
        print("[auto-format] Alpaca format detected.")
        def fmt(ex):
            return {"text": alpaca_prompt.format(
                ex.get("instruction","") or "",
                ex.get("input","") or "",
                ex.get("output","") or ""
            ) + EOS_TOKEN}
        return ds.map(fmt, remove_columns=ds.column_names)

    if {"prompt", "completion"}.issubset(cols):
        print("[auto-format] prompt/completion detected.")
        def fmt(ex):
            return {"text": alpaca_prompt.format(ex["prompt"], "", ex["completion"]) + EOS_TOKEN}
        return ds.map(fmt, remove_columns=ds.column_names)

    if "messages" in cols:
        print("[auto-format] messages format detected → applying Llama-3 chat template.")
        def fmt(ex):
            text = tokenizer.apply_chat_template(
                ex["messages"], tokenize=False, add_generation_prompt=False
            )
            return {"text": text}
        return ds.map(fmt, remove_columns=ds.column_names)

    if {"question", "answer"}.issubset(cols):
        print("[auto-format] question/answer detected.")
        def fmt(ex):
            return {"text": alpaca_prompt.format(ex["question"], "", ex["answer"]) + EOS_TOKEN}
        return ds.map(fmt, remove_columns=ds.column_names)

    raise ValueError(f"Unknown format. Columns: {ds.column_names}")

print("Helpers ready. Ab Step 4A chalao (file upload).")

### Step 4A — ⭐ Upload `.jsonl` (Choose Files button)

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

jsonl_paths = [name for name in uploaded.keys() if name.endswith((".jsonl", ".json"))]
if not jsonl_paths:
    raise ValueError("Koi .jsonl/.json file nahi mili!")

print("Uploaded:", jsonl_paths)
for p in jsonl_paths:
    print(f"  {p}  ({os.path.getsize(p)/1024/1024:.2f} MB)")

raw_dataset = load_dataset("json", data_files=jsonl_paths, split="train")
print(f"\nRaw: {len(raw_dataset)} rows | Columns: {raw_dataset.column_names}")
print("Sample:", raw_dataset[0])

dataset = auto_format_dataset(raw_dataset)
print(f"\nFormatted: {len(dataset)} rows")
print("Text preview:\n", dataset[0]["text"][:600])

### Step 4B — Path se load (Drive ya local) — skip if 4A done

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

JSONL_PATH = "/content/nexusai_train.jsonl"   # apni file ka path

raw_dataset = load_dataset("json", data_files=JSONL_PATH, split="train")
print(f"Raw: {len(raw_dataset)} rows | Columns: {raw_dataset.column_names}")

dataset = auto_format_dataset(raw_dataset)
print(f"\nFormatted: {len(dataset)} rows")
print("Text preview:\n", dataset[0]["text"][:600])

### Step 4C — HuggingFace public dataset — skip if 4A/4B done

In [ ]:
raw_dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = auto_format_dataset(raw_dataset)
print(f"Formatted: {len(dataset)} rows")
print("Text preview:\n", dataset[0]["text"][:600])

## Step 5 — Sanity check

In [ ]:
sample = dataset[0]["text"]
tokens = tokenizer(sample, return_tensors="pt")
print("Token count :", tokens.input_ids.shape[1])
print("First 20 ids:", tokens.input_ids[0][:20].tolist())
print("EOS present :", tokenizer.eos_token_id in tokens.input_ids[0].tolist())
print("\nLooks good!" if tokens.input_ids.shape[1] < max_seq_length else "\n⚠️ Sequence exceeds max_seq_length — will be truncated.")

## Step 6 — Configure SFTTrainer (Production settings)

**Key upgrades from demo:**
- `num_train_epochs=2` (full dataset coverage)
- `packing=True` (5x faster, ideal for chat-length data)
- `cosine` LR scheduler (smoother convergence)
- Checkpoints every 250 steps

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    args = SFTConfig(
        # --- Dataset ---
        dataset_text_field          = "text",
        max_seq_length              = max_seq_length,
        dataset_num_proc            = 2,
        packing                     = True,             # ⭐ 5x faster for short sequences
        padding_free                = False,
        # --- Training ---
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,                # effective batch = 8
        num_train_epochs            = 2,                # ⭐ full 2-epoch run
        warmup_ratio                = 0.05,             # ⭐ adaptive warmup
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",         # ⭐ smoother than linear
        seed                        = 3407,
        output_dir                  = "outputs",
        # --- Checkpoints ---
        save_steps                  = 250,              # ⭐ safety net
        save_total_limit            = 3,
        report_to                   = "none",
    ),
)

print(f"Trainer ready.")
print(f"Estimated steps: ~{len(dataset) * 2 // (2 * 4)} (with packing, actual may be less)")
print(f"Expected time on T4: ~1.5-2 hours")

## Step 7 — GPU memory snapshot

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory       = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU             = {gpu_stats.name}")
print(f"Max memory      = {max_memory} GB")
print(f"Reserved before = {start_gpu_memory} GB")
print(f"Available       = {round(max_memory - start_gpu_memory, 2)} GB")

## Step 8 — Train! 🚀

~1.5–2 hours on T4. Loss should drop from ~2.5 → ~0.8-1.2 over 2 epochs.

In [ ]:
trainer_stats = trainer.train()
print("\n✅ Training complete!")

## Step 9 — Training stats

In [ ]:
used_memory          = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage      = round(used_memory / max_memory * 100, 3)
lora_percentage      = round(used_memory_for_lora / max_memory * 100, 3)

print(f"Training time           = {trainer_stats.metrics['train_runtime']:.1f} sec")
print(f"                        = {trainer_stats.metrics['train_runtime']/60:.1f} min")
print(f"Final train loss        = {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"Peak reserved memory    = {used_memory} GB ({used_percentage}% of max)")
print(f"Training memory used    = {used_memory_for_lora} GB ({lora_percentage}% of max)")

## Step 10 — Inference (test NexusAI)

NexusAI ke actual use case pe test — prompt engineering request do.

In [ ]:
FastLanguageModel.for_inference(model)

# NexusAI style test — yahi tumhara model ka kaam hai
test_messages = [
    {"role": "system", "content": "You are NexusAI, an expert prompt engineer. Given a short raw idea from a user, produce a high-quality, structured prompt suitable for the right AI tool."},
    {"role": "user", "content": "write a blog post about AI in healthcare for beginners\n(skill level: beginner)"},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids      = inputs,
    max_new_tokens = 512,
    use_cache      = True,
    temperature    = 0.7,
    top_p          = 0.9,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("NexusAI Response:")
print("=" * 60)
print(response)

### Step 10b — More test prompts

In [ ]:
# Aur test cases
test_prompts = [
    "generate a logo for a coffee shop using midjourney\n(skill level: intermediate)",
    "explain quantum computing to a 10 year old\n(skill level: beginner)",
    "write python code to scrape stock prices\n(skill level: pro)",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are NexusAI, an expert prompt engineer. Given a short raw idea from a user, produce a high-quality, structured prompt suitable for the right AI tool."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"USER: {prompt}")
    print(f"{'='*60}")
    print(response)
    print()

## Step 11 — Save model

3 options. Default = LoRA only (fast, small, ~200MB).

In [ ]:
# Option 1: LoRA adapters only (recommended)
model.save_pretrained("nexusai_lora")
tokenizer.save_pretrained("nexusai_lora")
print("✅ LoRA saved → ./nexusai_lora")

# Option 2: Full merged 16-bit model (uncomment)
# model.save_pretrained_merged("nexusai_merged_16bit", tokenizer, save_method="merged_16bit")
# print("✅ Merged model saved → ./nexusai_merged_16bit")

# Option 3: Push to HuggingFace Hub (uncomment + add your token)
# model.push_to_hub("your-username/NexusAI-lora", token="hf_xxx")
# tokenizer.push_to_hub("your-username/NexusAI-lora", token="hf_xxx")
# print("✅ Pushed to HuggingFace Hub")

## Step 12 — Reload and verify saved model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "nexusai_lora",
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)
FastLanguageModel.for_inference(model)

# Chat template wapas set karo (reload ke baad missing ho sakta hai)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

# Final verification
messages = [
    {"role": "system", "content": "You are NexusAI, an expert prompt engineer."},
    {"role": "user", "content": "create a marketing email for a SaaS product launch\n(skill level: pro)"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.7)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("=" * 60)
print("NexusAI (reloaded) Response:")
print("=" * 60)
print(response)
print("\n🎉 NexusAI production fine-tune COMPLETE!")